<a href="https://colab.research.google.com/github/Rut092/rut-ai-portfolio-PHASE2-DL/blob/main/Guard_rails(batchnorm%2Cdropout%2Cl2_reg).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
import torch
import torch.nn as nn
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader,TensorDataset
import time
import copy

In [33]:
class RobustClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        # Core Layers
        self.layer1 = nn.Linear(in_features = 2, out_features = 32)
        self.layer2 = nn.Linear(in_features=32,out_features = 16)
        self.output_layer = nn.Linear(in_features = 16,out_features=1)

        # Normalization
        # we specify no. of features it neeed to normalize
        self.batchnorm1 = nn.BatchNorm1d(num_features = 32)
        self.batchnorm2 = nn.BatchNorm1d(num_features = 16)

        # Regularization
        self.dropout = nn.Dropout(p=0.3)

        # Activation Function
        self.relu = nn.ReLU()

    def forward(self,x):
        # flow
        x = self.layer1(x)
        x = self.batchnorm1(x)
        x = self.relu(x)
        x = self.dropout(x)

        # flow 2
        x = self.layer2(x)
        x = self.batchnorm2(x)
        x = self.relu(x)
        x = self.dropout(x)

        # outflow - output logits directly
        x = self.output_layer(x)

        return x

In [34]:
model = RobustClassifier()
print(model)

RobustClassifier(
  (layer1): Linear(in_features=2, out_features=32, bias=True)
  (layer2): Linear(in_features=32, out_features=16, bias=True)
  (output_layer): Linear(in_features=16, out_features=1, bias=True)
  (batchnorm1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (batchnorm2): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (relu): ReLU()
)


In [35]:
X, y = make_moons(n_samples = 1000, noise=0.4, random_state=42)
X_train,X_test,y_train,y_test =  train_test_split(X,y, test_size = 0.2, random_state = 42,stratify=y)

print(X_train.shape,y_train.shape)

(800, 2) (800,)


In [36]:
X_train_tensor = torch.from_numpy(X_train).float()
X_test_tensor = torch.from_numpy(X_test).float()

y_train = torch.from_numpy(y_train).reshape(-1,1)
y_test = torch.from_numpy(y_test).reshape(-1,1)

In [37]:
epochs = 10
learning_rate = 0.001
optimizer = torch.optim.AdamW(model.parameters(),lr = learning_rate,weight_decay= 0.1)
criterion = nn.BCEWithLogitsLoss()
batch_size = 10


In [38]:
from numpy import float32
train_dataset = TensorDataset(X_train_tensor, y_train.to(torch.float32))
test_dataset = TensorDataset(X_test_tensor, y_test.to(torch.float32))

dataset_sizes = {
    'train': len(train_dataset),
    'test' : len(test_dataset)
}

dataloaders = {
    'train': DataLoader(train_dataset, batch_size = batch_size,shuffle = True),
    'test': DataLoader(test_dataset, batch_size = batch_size,shuffle = False)
}

In [39]:
def train_model(model,criterion,optimizer,num_epochs, device = 'cpu'):
    since = time.time()

    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs-1}')
        print('-'*10)

        for phase in ['train','test']:
            if phase =='train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_correct = 0.0

            for inputs,labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase=='train'):
                    outputs = model(inputs)
                    loss = criterion(outputs,labels)

                    if phase=='train':
                        loss.backward()
                        optimizer.step()

                running_loss+=loss.item()*inputs.size(0)
                # Apply sigmoid to logits to get probabilities, then threshold for binary predictions
                preds = (torch.sigmoid(outputs) >= 0.5).float()
                running_correct += torch.sum(preds == labels).item()

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_correct / dataset_sizes[phase]

            print(f'{phase} Loss: {epoch_loss} Acc: {epoch_acc}')

            if phase =='test' and epoch_acc>best_acc:
                best_model_wts = copy.deepcopy(model.state_dict())
                best_acc = epoch_acc

        print()


    time_elapsed = time.time() - since
    print(f'training completed in {time_elapsed //60:.0f}m {time_elapsed %60:.0f}s')

    print(f'Best Test Accuracy: {best_acc:.4f}')


In [40]:
model = model.to('cpu')

In [41]:
train_model(model,criterion,optimizer,epochs)

Epoch 0/9
----------
train Loss: 0.6098073355853557 Acc: 0.68875
test Loss: 0.4683684498071671 Acc: 0.905

Epoch 1/9
----------
train Loss: 0.5275267582386732 Acc: 0.7725
test Loss: 0.4088346615433693 Acc: 0.91

Epoch 2/9
----------
train Loss: 0.4863495983183384 Acc: 0.77375
test Loss: 0.370758144557476 Acc: 0.91

Epoch 3/9
----------
train Loss: 0.47151546869426963 Acc: 0.7975
test Loss: 0.3498841434717178 Acc: 0.895

Epoch 4/9
----------
train Loss: 0.4597268901765347 Acc: 0.78875
test Loss: 0.3474936172366142 Acc: 0.905

Epoch 5/9
----------
train Loss: 0.44226926900446417 Acc: 0.80125
test Loss: 0.33731610476970675 Acc: 0.91

Epoch 6/9
----------
train Loss: 0.451673392765224 Acc: 0.785
test Loss: 0.32478565238416196 Acc: 0.905

Epoch 7/9
----------
train Loss: 0.4593937791883945 Acc: 0.8025
test Loss: 0.32842412367463114 Acc: 0.91

Epoch 8/9
----------
train Loss: 0.477452627196908 Acc: 0.795
test Loss: 0.31994012035429475 Acc: 0.91

Epoch 9/9
----------
train Loss: 0.45192813463